💡 **Environment:** `clamp-analyses` (R kernel)

# Figure 4: projection into the ARCHS4 model and drug-disease prediction

Exported at **180 × 170 mm** with all text at least **5 pt** (Nature Methods maximum).

- **a**: projection schematic, `scripts/panels/assets/projection_scheme.svg` (a copy of the manuscript's
  `content/images/final_figures/projection_scheme.svg`; re-copy it after editing). It is rasterized at 600 dpi
  with Inkscape because the R environment has no SVG importer.
- **b**: IFN-alpha EndoC-betaH1 curated mechanisms, ARCHS4 projection versus the dataset-trained (Cyt) model.
  The matching gene-rank comparison is in Supplementary Figure 6d.
- **c**: drug-disease prediction, three CLAMP models and the gene-based approach (fixed-threshold scoring).

In [ ]:
source(here::here("scripts/panels/main_figure.R"))
source(here("scripts/archs4/projections/plots.R"))
suppressPackageStartupMessages({ library(png) })
set.seed(1)
dirs <- main_figure_dirs("fig4")
source_dir <- dirs$source
FIG_W <- 180
FIG_H <- 170

fmt_sci <- function(p) {
  if (p >= .001) return(sprintf("%.3f", p))
  e <- floor(log10(p))
  sup <- c("⁰", "¹", "²", "³", "⁴", "⁵", "⁶", "⁷", "⁸", "⁹")
  paste0(sprintf("%.1f", p / 10^e), "×10⁻", paste0(sup[as.integer(strsplit(as.character(abs(e)), "")[[1]]) + 1], collapse = ""))
}

## Panel a: projection schematic

In [ ]:
scheme_svg <- here("scripts/panels/assets/projection_scheme.svg")
scheme_png <- file.path(source_dir, "a_projection_scheme.png")
stopifnot(file.exists(scheme_svg), nzchar(Sys.which("inkscape")))
status <- system2("inkscape", c(shQuote(scheme_svg), "--export-type=png", "--export-dpi=600",
                                "--export-background=white", paste0("--export-filename=", shQuote(scheme_png))),
                  stdout = FALSE, stderr = FALSE)
stopifnot(status == 0L, file.exists(scheme_png))
scheme <- png::readPNG(scheme_png)
SCHEME_ASPECT <- dim(scheme)[1] / dim(scheme)[2]
plot_scheme <- ggdraw() + draw_grob(grid::rasterGrob(scheme, interpolate = TRUE))

## Panel b: curated IFN-alpha mechanisms in the ARCHS4 projection and the dataset-trained model

In [ ]:
agg_dir <- file.path(DATA_DIR, "03_projections", "aggregate")
dataset_b <- "cyt_ifna_GSE133218"
lv_stats <- fread(file.path(agg_dir, "lv_stats_long.csv"))[group == "cytokines"]
# Curated cytokine mechanisms and their matched modules, as in 00_cytokines.executed.ipynb
# (Supplementary Table 2 lists every target of the three external datasets).
curated <- data.table(
  mechanism = rep(c("IFN-alpha response", "IRF/STAT response", "MHC-I presentation", "ER stress",
                    "Protein mod./degradation", "Alternative splicing", "PD-L1/HLA-E protection",
                    "Beta cells"), each = 2L),
  model = rep(c("ARCHS4", "local"), 8L),
  LV = c("LV66", "LV23", "LV66", "LV23", "LV882", "LV29", "LV134", NA, "LV1269", NA, "LV49", NA,
         "LV8", "LV9", "LV63", "LV13"),
  gene_set = c("HALLMARK_INTERFERON_ALPHA_RESPONSE", "HALLMARK_INTERFERON_ALPHA_RESPONSE",
               "HALLMARK_INTERFERON_ALPHA_RESPONSE", "HALLMARK_INTERFERON_ALPHA_RESPONSE",
               "KEGG_MEDICUS_REFERENCE_ANTIGEN_PROCESSING_AND_PRESENTATION_BY_MHC_CLASS_I_MOLECULES",
               "KEGG_MEDICUS_REFERENCE_ANTIGEN_PROCESSING_AND_PRESENTATION_BY_MHC_CLASS_I_MOLECULES",
               "REACTOME_ATF4_ACTIVATES_GENES_IN_RESPONSE_TO_ENDOPLASMIC_RETICULUM_STRESS", NA,
               "KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION", NA,
               "REACTOME_MRNA_SPLICING", NA,
               "REACTOME_REGULATION_OF_PD_L1_CD274_TRANSCRIPTION", "REACTOME_REGULATION_OF_PD_L1_CD274_TRANSCRIPTION",
               "HALLMARK_PANCREAS_BETA_CELLS", "HALLMARK_PANCREAS_BETA_CELLS"),
  ora_fdr = c(2.368199e-84, 1.790997e-32, 2.368199e-84, 1.790997e-32, 9.721743e-12, 2.953455e-09,
              5.508900e-08, NA, 2.726521e-59, NA, 5.268816e-04, NA, 9.214699e-04, 5.009466e-03,
              3.733612e-08, 2.994879e-08))
curated[, `:=`(dataset = dataset_b, recovered = !is.na(LV))]
fwrite(curated, file.path(source_dir, "b_curated_mechanisms.csv"))

contrasts <- c("tp_2h", "tp_8h", "tp_24h")
model_labels <- c(ARCHS4 = "ARCHS4 projection", local = "Cyt model")
pts <- merge(curated[recovered == TRUE], lv_stats[dataset == dataset_b & contrast %in% contrasts],
             by = c("dataset", "model", "LV"), allow.cartesian = TRUE)
fwrite(pts, file.path(source_dir, "b_mechanism_points.csv"))
mech_levels <- rev(unique(curated$mechanism))
pts[, `:=`(mechanism = factor(mechanism, levels = mech_levels), contrast = factor(contrast, levels = contrasts),
           model = factor(model_labels[model], levels = model_labels))]
missing <- curated[recovered == FALSE]
missing[, `:=`(mechanism = factor(mechanism, levels = mech_levels),
               model = factor(model_labels[model], levels = model_labels),
               contrast = factor("tp_8h", levels = contrasts))]

# The two models are not calibrated to the same module log2FC range, so each keeps its own
# fill scale; the -log10 ORA FDR size scale is shared and drawn once.
size_range <- range(-log10(pmax(pts$ora_fdr, 1e-300)))
dot_panel <- function(label) {
  dd <- pts[model == label]
  lim <- max(abs(scales::breaks_pretty(n = 3)(range(c(-abs(dd$logFC), abs(dd$logFC))))))
  ggplot(dd, aes(contrast, mechanism)) +
    geom_point(aes(size = -log10(pmax(ora_fdr, 1e-300)), fill = logFC), shape = 21, stroke = .2) +
    geom_text(data = missing[model == label], aes(contrast, mechanism), label = "Not recovered",
              inherit.aes = FALSE, size = 5 / .pt, colour = "grey40", fontface = "italic") +
    scale_size_continuous(range = c(1.2, 3.8), name = "−log10 ORA FDR", breaks = c(10, 40, 80), limits = size_range) +
    scale_fill_gradient2(low = "#1a9850", mid = "white", high = "#d73027", name = paste0("Module log2FC, ", c("ARCHS4 projection" = "ARCHS4", "Cyt model" = "Cyt model")[[label]]),
                         limits = c(-lim, lim), breaks = c(-lim, 0, lim),
                         labels = scales::label_number(accuracy = if (lim < 1) .1 else 1)) +
    scale_x_discrete(labels = function(x) sub("tp_", "", x), drop = FALSE) +
    scale_y_discrete(drop = FALSE) +
    labs(x = "IFN-alpha exposure", y = NULL, title = label) +
    theme_nm() +
    theme(legend.position = "bottom", legend.box = "vertical", legend.key.height = unit(1.2, "mm"),
          legend.key.width = unit(3, "mm"), legend.margin = margin(0, 0, 0, 0),
          plot.title = element_text(size = FS_TITLE, face = "bold", hjust = .5))
}
# Each fill colour bar sits under its own dot panel (ggplot centres a bottom legend on the panel)
# with a centred title; the shared size legend is centred across both panels via patchwork.
size_legend <- get_legend(dot_panel(model_labels[["ARCHS4"]]) + guides(fill = "none"))
fill_guide <- list(
  guides(size = "none", fill = guide_colourbar(title.position = "top", title.hjust = 0.5,
                                               barwidth = unit(14, "mm"), barheight = unit(.8, "mm"))),
  theme(legend.title = element_text(hjust = 0.5)))
dots_a <- dot_panel(model_labels[["ARCHS4"]]) + fill_guide
dots_l <- dot_panel(model_labels[["local"]]) + fill_guide +
  theme(axis.text.y = element_blank(), axis.ticks.y = element_blank())
plot_b_pw <- patchwork::wrap_plots(dots_a, dots_l, patchwork::wrap_elements(panel = size_legend),
                                   design = "AB\nCC", widths = c(1, 1), heights = c(.88, .12))
plot_b <- patchwork::patchworkGrob(plot_b_pw)
print(plot_b)

## Panel c: drug-disease prediction

In [ ]:
dd_dir <- file.path(DATA_DIR, "02_drug_diseases_canonical", "three_model_comparison", "data")
dd_order <- c("ARCHS4", "recount2", "GTEx", "Gene-based")
dd_colors <- COMPENDIUM_COLORS[dd_order]  # config.yaml COMPENDIUM_COLORS
dominance <- fread(file.path(dd_dir, "ordering_stability.csv"))
tissue <- fread(file.path(dd_dir, "per_tissue_metrics.csv"))
agg_ref <- fread(file.path(dd_dir, "max_aggregate_reference.csv"))
tests <- fread(file.path(dd_dir, "paired_tissue_tests.csv"))
stopifnot(uniqueN(tissue$tissue) == 49, setequal(tissue$method, dd_order))
for (nm in c("ordering_stability", "per_tissue_metrics", "max_aggregate_reference", "paired_tissue_tests"))
  fwrite(fread(file.path(dd_dir, paste0(nm, ".csv"))), file.path(source_dir, paste0("c_", nm, ".csv")))

dom <- dominance[row_m != col_m]
dom[, `:=`(row_m = factor(row_m, levels = dd_order), col_m = factor(col_m, levels = dd_order))]
plot_c1 <- ggplot(dom, aes(col_m, row_m, fill = value)) +
  geom_tile(colour = "white") +
  geom_text(aes(label = sprintf("%.2f", value), colour = value >= 0.7), size = 5 / .pt, fontface = "bold",
            show.legend = FALSE) +
  scale_fill_gradient(low = "white", high = "#1B9E77", limits = c(0, 1), name = "P(row >\ncolumn)",
                      breaks = c(0, .5, 1), labels = c("0", ".5", "1"),
                      guide = guide_colourbar(barwidth = unit(2, "mm"), barheight = unit(12, "mm"))) +
  scale_colour_manual(values = c("TRUE" = "white", "FALSE" = "black")) +
  scale_x_discrete(position = "top", labels = function(x) ifelse(x == "Gene-based", "Gene-\nbased", x)) +
  coord_fixed() +
  labs(x = NULL, y = NULL) +
  theme_nm() +
  theme(axis.line = element_blank(), axis.ticks = element_blank(), axis.text.x = element_text(angle = 30, hjust = 0),
        legend.position = "right", legend.title.position = "top",
        legend.key.width = unit(2, "mm"), legend.key.height = unit(2, "mm"))

tissue[, method := factor(method, levels = dd_order)]
agg_ref[, method := factor(method, levels = dd_order)]
top_y <- max(tissue$auroc, agg_ref$auroc)
tests_ord <- tests[group1 == "ARCHS4" | group2 == "ARCHS4"]
tests_ord[, `:=`(xstart = match(group1, dd_order), xend = match(group2, dd_order))]
tests_ord <- tests_ord[order(xend - xstart)]
plot_c2 <- ggplot(tissue, aes(method, auroc)) +
  geom_violin(aes(fill = method), width = 0.7, trim = TRUE, scale = "width", alpha = 0.65, linewidth = 0.25, show.legend = FALSE) +
  geom_boxplot(aes(fill = method), width = 0.18, outlier.shape = NA, alpha = 0.55, linewidth = 0.25, show.legend = FALSE) +
  geom_jitter(aes(colour = method), width = 0.08, size = 0.4, alpha = 0.5, show.legend = FALSE) +
  geom_point(data = agg_ref, aes(method, auroc, shape = "Maximum over\ntissues"), size = 1.5, fill = "white",
             colour = "black", stroke = 0.4) +
  scale_fill_manual(values = dd_colors) +
  scale_colour_manual(values = dd_colors) +
  scale_shape_manual(values = c("Maximum over\ntissues" = 23), name = NULL) +
  labs(x = NULL, y = "AUROC (per GTEx tissue)") +
  coord_cartesian(ylim = c(min(tissue$auroc, agg_ref$auroc) - 0.012, top_y + 0.10)) +
  theme_nm() +
  theme(legend.position = "right", legend.margin = margin(0, 0, 0, 0),
        axis.text.x = element_text(angle = 30, hjust = 1))
for (i in seq_len(nrow(tests_ord))) {
  y <- top_y + 0.012 + 0.022 * i
  plot_c2 <- plot_c2 +
    annotate("segment", x = tests_ord$xstart[i], xend = tests_ord$xend[i], y = y, yend = y, linewidth = 0.25) +
    annotate("text", x = (tests_ord$xstart[i] + tests_ord$xend[i]) / 2, y = y + 0.003,
             label = sprintf("q = %s", fmt_sci(tests_ord$q_value_bh[i])), size = 5 / .pt, vjust = 0)
}
# Stack c1 over c2 so each plot has room for its labels and a legend at the right.
plot_c <- plot_grid(plot_c1, plot_c2, ncol = 1, rel_heights = c(0.85, 1.15), align = "v")
print(plot_c)

## Assemble and save

In [ ]:
scheme_h <- FIG_W * SCHEME_ASPECT
row2 <- plot_grid(add_tag(plot_b, "b", top = 0.96), add_tag(plot_c, "c", top = 0.96), nrow = 1, rel_widths = c(1.05, 1))
fig4 <- plot_grid(add_tag(plot_scheme, "a", top = 0.97), row2, ncol = 1,
                  rel_heights = c(scheme_h + 3, FIG_H - scheme_h - 3))
writeLines(c(sprintf("Figure 4: %d x %d mm; minimum text 5 pt; schematic %.1f mm tall.", FIG_W, FIG_H, scheme_h),
             "b: aggregate/lv_stats_long.csv with curated matches from 00_cytokines.executed.ipynb.",
             "c: 02_drug_diseases_canonical/three_model_comparison/data (top 10 modules or 100 genes; 49 tissues)."),
           file.path(source_dir, "README.txt"))
save_main_figure(fig4, "fig4", FIG_W, FIG_H)
options(repr.plot.width = FIG_W / 25.4, repr.plot.height = FIG_H / 25.4)
print(fig4)